# Data and Experimental Design Overview

## Objective

Describe the sci-Plex2 A549 dataset and the experimental design that underlies the downstream perturbation analysis.

## Data

A focused A549 / four-compound sci-Plex2 subset was selected for a tractable perturbation analysis (Srivatsan et al., *Science*, 2020). Data are stored as an AnnData object; the relevant components are:

| Slot | Content |
|---|---|
| `.X` | cell × gene expression matrix (raw UMI counts) |
| `.obs` | per-cell metadata: `perturbation`, `dose_value`, `well`, `cell_line`, QC metrics |
| `.var` | per-gene metadata |
| `.layers` | additional matrices (raw counts preserved) |
| `.obsm` | cell embeddings (PCA / UMAP) |

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import pertpy as pt


In [2]:
adata = pt.dt.srivatsan_2020_sciplex2()


## Cell and gene counts

In [3]:
print("n_obs (cells) =", adata.n_obs)
print("n_vars (genes) =", adata.n_vars)
print("shape           =", adata.shape)


n_obs (cells) = 24262
n_vars (genes) = 58347
shape           = (24262, 58347)


## Raw-count evidence

`.X` contains integer-valued UMI counts rather than normalized expression, verified by integer values, a non-negative bounded range, and row sums that match the provided `ncounts` column.

In [4]:
print("type  :", type(adata.X).__name__)
print("dtype :", adata.X.dtype)
print("shape  :", adata.X.shape)

print("first cell, first 5 genes:", adata.X[0, :5].toarray())


type  : csr_matrix
dtype : float32
shape  : (24262, 58347)
first cell, first 5 genes: [[0. 0. 0. 0. 0.]]


## Experimental design

The dataset contains four compounds plus a vehicle control across eight dose levels (0–100 µM).

In [5]:
adata.obs['perturbation'].value_counts()


perturbation
Dex        8064
Nutlin     5956
SAHA       5530
BMS        4183
control     529
Name: count, dtype: int64

In [6]:
doses = sorted(adata.obs['dose_value'].dropna().unique(), key=lambda x: float(x))
print(doses)


['0', '0.1', '0.5', '1', '5', '10', '50', '100']


## Replicate and control structure

There is no explicit replicate column; the well is the highest available experimental unit (six wells per drug–dose condition). The vehicle control (`perturbation == 'control'`) lacks dose, well, and hashtag-oligo annotations.

In [7]:
n_wells = (
    adata.obs
    .dropna(subset=['dose_value'])
    .groupby(['perturbation', 'dose_value'], observed=True)['well']
    .nunique()
)
print("drug x dose combinations =", len(n_wells))
print("wells per combination (should all be equal):", n_wells.unique())
print("total wells =", adata.obs['well'].nunique(dropna=True))


drug x dose combinations = 32
wells per combination (should all be equal): [6]
total wells = 192


In [8]:
ctrl = adata.obs[adata.obs['perturbation'] == 'control']
print("control cells =", len(ctrl))
print("control  dose_value all NaN:", ctrl['dose_value'].isna().all())
print("control  well all NaN       :", ctrl['well'].isna().all())
print("control  top_oligo all NaN  :", ctrl['top_oligo'].isna().all())


control cells = 529
control  dose_value all NaN: True
control  well all NaN       : True
control  top_oligo all NaN  : True


## Experimental Design Summary

- **24,262 cells × 58,347 genes** (raw UMI counts).
- **4 compounds** (dexamethasone, Nutlin-3a, BMS-345541, vorinostat/SAHA) + vehicle control.
- **8 dose levels** (0–100 µM).
- **6 wells per drug–dose condition** (192 wells total).
- The un-hashed control lacks dose / well / oligo annotations.
- Raw counts are integer-valued and match the provided `ncounts` column.

## Save local working copy

The loaded AnnData object is written to `data/srivatsan_2020_sciplex2.h5ad` (only if not already present) so that subsequent notebooks can read it directly without re-downloading.

In [9]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    if (PROJECT_ROOT.parent / "README.md").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw_path = DATA_DIR / "srivatsan_2020_sciplex2.h5ad"
if not raw_path.exists():
    adata.write_h5ad(raw_path)
    print(f"saved {raw_path.relative_to(PROJECT_ROOT)}")
else:
    print(f"already present: {raw_path.relative_to(PROJECT_ROOT)}")


already present: data\srivatsan_2020_sciplex2.h5ad
